In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    RocCurveDisplay, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, classification_report,
)


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 0


In [ ]:
churn = pd.read_csv(DATA / "churn.csv")
houses = pd.read_csv(DATA / "house_prices.csv")


In [ ]:
print("churn:", churn.shape)
print("houses:", houses.shape)


In [ ]:
churn.head()


In [ ]:
# ======================
# 2. CHURN: МОДЕЛЬ
# ======================
num_cols = ["tenure_months", "monthly_charges", "support_tickets", "total_charges"]
cat_cols = ["contract_type", "internet_service"]

X = churn[num_cols + cat_cols]
y = churn["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)


In [ ]:
clf = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
    ("lr", LogisticRegression(max_iter=500)),
])
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))


In [ ]:
# ======================
# 3. ROC-КРИВАЯ
# ======================
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(clf, X_test, y_test, ax=ax, name="LogReg")
ax.set_title("Churn — ROC-кривая", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "churn_roc.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 4. CONFUSION MATRIX
# ======================
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Churn — confusion matrix")
plt.tight_layout()
plt.savefig(FIG / "churn_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Распределение вероятностей по классам
plt.figure(figsize=(8, 5))
sns.histplot(proba[y_test == 0], bins=25, color="#440154", alpha=0.6, label="churn=0", kde=True)
sns.histplot(proba[y_test == 1], bins=25, color="#fde725", alpha=0.6, label="churn=1", kde=True)
plt.axvline(0.5, color="red", linestyle="--", label="порог 0.5")
plt.title("Распределение вероятностей churn", fontsize=14, pad=15)
plt.xlabel("P(churn=1)", fontsize=12)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "churn_proba_hist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 5. HOUSES: РЕГРЕССИЯ И ОСТАТКИ
# ======================
Xh = houses[["area_sqm", "rooms", "dist_center_km", "year_built", "floor", "parking"]]
yh = houses["price"]

Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=RANDOM_SEED)

reg = Pipeline([("sc", StandardScaler()), ("lr", LinearRegression())])
reg.fit(Xh_tr, yh_tr)
pred_h = reg.predict(Xh_te)
residuals = yh_te - pred_h

print("R2 holdout:", round(reg.score(Xh_te, yh_te), 3))


In [ ]:
# Остатки
plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred_h, y=residuals, alpha=0.4, s=25, color="#31688e")
plt.axhline(0, color="red", linestyle="--", lw=1.5)
plt.title("Остатки линейной регрессии (houses)", fontsize=14, pad=15)
plt.xlabel("Прогноз", fontsize=12)
plt.ylabel("Остаток (y − ŷ)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_residuals.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Гистограмма остатков
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=35, kde=True, color="#35b779", edgecolor="white")
plt.title("Распределение остатков", fontsize=14, pad=15)
plt.xlabel("Остаток", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_residuals_hist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Факт vs прогноз
plt.figure(figsize=(6, 6))
sns.scatterplot(x=yh_te, y=pred_h, alpha=0.4, s=25, color="#21918c")
lims = [min(yh_te.min(), pred_h.min()), max(yh_te.max(), pred_h.max())]
plt.plot(lims, lims, "r--", lw=1.2)
plt.title("Houses: факт vs прогноз", fontsize=14, pad=15)
plt.xlabel("Факт", fontsize=12)
plt.ylabel("Прогноз", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / "houses_diag_pred_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Диагностика: ROC/CM для классификации, остатки — для регрессии.")
